In [30]:
"""
Customer Segmentation Analysis - Part 3: Market Basket & Recommendations
========================================================================

This notebook provides:
1. Market basket analysis and association rules
2. Segment-specific product recommendations
3. Cross-selling and upselling opportunities
4. Final business recommendations
"""

'\nCustomer Segmentation Analysis - Part 3: Market Basket & Recommendations\n========================================================================\n\nThis notebook provides:\n1. Market basket analysis and association rules\n2. Segment-specific product recommendations\n3. Cross-selling and upselling opportunities\n4. Store-level insights\n5. Final business recommendations\n'

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [32]:
# Load data
print("Loading data...")
df_transactions = pd.read_csv('TOKEN_202401142117.csv').merge(
    pd.read_csv('VENTE_202401142111.csv'), on='CD_TICKET_UNIQUE'
).merge(
    pd.read_csv('ARTICLE_202401142118.csv'), on='ID_ARTICLE', how='left'
)

customer_segments = pd.read_csv('customers_with_segments.csv')

print(f"✓ Transaction data loaded: {df_transactions.shape}")
print(f"✓ Customer segments loaded: {customer_segments.shape}")

Loading data...
✓ Transaction data loaded: (749565, 18)
✓ Customer segments loaded: (48309, 27)


In [33]:
# ============================================================================
# PART 7: MARKET BASKET ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("PART 7: MARKET BASKET ANALYSIS")
print("=" * 80)


PART 7: MARKET BASKET ANALYSIS


In [34]:
# ============================================================================
# STEP 7.1: Prepare Transaction Baskets
# ============================================================================

print("\n[STEP 7.1] Preparing transaction baskets...")

# Create baskets at family level (more manageable than individual products)
baskets = df_transactions.groupby(['CD_TICKET_UNIQUE', 'LB_FAMILLE'])['QT_UVC'].sum().reset_index()
baskets = baskets[baskets['LB_FAMILLE'].notna()]

# Create basket format for association rules
basket_items = baskets.groupby('CD_TICKET_UNIQUE')['LB_FAMILLE'].apply(list).reset_index()

print(f"✓ Created {len(basket_items)} baskets")
print(f"  Average items per basket: {baskets.groupby('CD_TICKET_UNIQUE').size().mean():.2f}")

# Sample of baskets
print("\nSample baskets:")
for i in range(min(5, len(basket_items))):
    print(f"  Basket {i+1}: {basket_items.iloc[i]['LB_FAMILLE'][:5]}")



[STEP 7.1] Preparing transaction baskets...
✓ Created 54423 baskets
  Average items per basket: 6.12

Sample baskets:
  Basket 1: ['Oeufs']
  Basket 2: ['Dessert']
  Basket 3: ['Coquillages', 'Poissons']
  Basket 4: ['Oeufs']
  Basket 5: ['Dessert']


In [35]:
# ============================================================================
# STEP 7.2: Generate Association Rules - Overall
# ============================================================================

print("\n[STEP 7.2] Generating association rules (overall)...")

# Convert to one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(basket_items['LB_FAMILLE']).transform(basket_items['LB_FAMILLE'])
basket_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"✓ One-hot encoded baskets: {basket_encoded.shape}")

# Find frequent itemsets
print("  Finding frequent itemsets...")
frequent_itemsets = apriori(basket_encoded, min_support=0.01, use_colnames=True)
print(f"✓ Found {len(frequent_itemsets)} frequent itemsets")

# Generate association rules
print("  Generating association rules...")
if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
    rules = rules.sort_values('lift', ascending=False)
    
    print(f"✓ Generated {len(rules)} association rules")
    
    # Display top rules
    print("\nTop 10 Association Rules (by Lift):")
    print("=" * 120)
    top_rules = rules.head(10)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    
    for idx, row in top_rules.iterrows():
        ant = list(row['antecedents'])[0] if len(row['antecedents']) == 1 else row['antecedents']
        cons = list(row['consequents'])[0] if len(row['consequents']) == 1 else row['consequents']
        print(f"\n{ant}")
        print(f"  → {cons}")
        print(f"  Support: {row['support']:.3f} | Confidence: {row['confidence']:.3f} | Lift: {row['lift']:.2f}")
    
    # Save rules
    rules.to_csv('association_rules_overall.csv', index=False)
    print("\n✓ Association rules saved to 'association_rules_overall.csv'")
else:
    print("⚠ No frequent itemsets found. Consider lowering min_support threshold.")
    rules = pd.DataFrame()



[STEP 7.2] Generating association rules (overall)...
✓ One-hot encoded baskets: (54423, 113)
  Finding frequent itemsets...
✓ Found 2536 frequent itemsets
  Generating association rules...
✓ Generated 15642 association rules

Top 10 Association Rules (by Lift):

frozenset({'Courgette', 'Poivron'})
  → frozenset({'Aubergine', 'Tomate'})
  Support: 0.013 | Confidence: 0.305 | Lift: 7.91

frozenset({'Aubergine', 'Tomate'})
  → frozenset({'Courgette', 'Poivron'})
  Support: 0.013 | Confidence: 0.341 | Lift: 7.91

Aubergine
  → frozenset({'Courgette', 'Poivron', 'Tomate'})
  Support: 0.013 | Confidence: 0.217 | Lift: 7.14

frozenset({'Courgette', 'Poivron', 'Tomate'})
  → Aubergine
  Support: 0.013 | Confidence: 0.432 | Lift: 7.14

Aubergine
  → frozenset({'Courgette', 'Poivron'})
  Support: 0.018 | Confidence: 0.289 | Lift: 6.70

frozenset({'Courgette', 'Poivron'})
  → Aubergine
  Support: 0.018 | Confidence: 0.406 | Lift: 6.70

frozenset({'Poivron', 'Tomate'})
  → frozenset({'Condiment',

In [36]:
# ============================================================================
# STEP 7.3: Segment-Specific Association Rules
# ============================================================================

print("\n[STEP 7.3] Generating segment-specific association rules...")

# Merge segments with transactions
df_trans_segments = df_transactions.merge(
    customer_segments[['NO_TOKEN_CB', 'Cluster']], 
    on='NO_TOKEN_CB', 
    how='left'
)

# Generate rules for each segment
segment_rules_dict = {}

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    print(f"\n  Analyzing Segment {cluster_id}...")
    
    # Get transactions for this segment
    segment_trans = df_trans_segments[df_trans_segments['Cluster'] == cluster_id]
    
    # Create baskets
    segment_baskets = segment_trans.groupby(['CD_TICKET_UNIQUE', 'LB_FAMILLE'])['QT_UVC'].sum().reset_index()
    segment_baskets = segment_baskets[segment_baskets['LB_FAMILLE'].notna()]
    
    if len(segment_baskets) < 10:
        print(f"    ⚠ Insufficient data for Segment {cluster_id}")
        continue
    
    segment_basket_items = segment_baskets.groupby('CD_TICKET_UNIQUE')['LB_FAMILLE'].apply(list).reset_index()
    
    # Encode and find patterns
    try:
        te_seg = TransactionEncoder()
        te_seg_ary = te_seg.fit(segment_basket_items['LB_FAMILLE']).transform(segment_basket_items['LB_FAMILLE'])
        basket_seg_encoded = pd.DataFrame(te_seg_ary, columns=te_seg.columns_)
        
        freq_items_seg = apriori(basket_seg_encoded, min_support=0.02, use_colnames=True)
        
        if len(freq_items_seg) > 0:
            rules_seg = association_rules(freq_items_seg, metric="lift", min_threshold=1.0)
            rules_seg = rules_seg.sort_values('lift', ascending=False)
            segment_rules_dict[cluster_id] = rules_seg
            
            print(f"    ✓ Found {len(rules_seg)} rules for Segment {cluster_id}")
            
            # Save segment rules
            rules_seg.to_csv(f'association_rules_segment_{cluster_id}.csv', index=False)
        else:
            print(f"    ⚠ No rules found for Segment {cluster_id}")
    except Exception as e:
        print(f"    ⚠ Error processing Segment {cluster_id}: {str(e)}")



[STEP 7.3] Generating segment-specific association rules...

  Analyzing Segment 0...
    ✓ Found 810 rules for Segment 0

  Analyzing Segment 1...
    ✓ Found 242164 rules for Segment 1

  Analyzing Segment 2...
    ✓ Found 654 rules for Segment 2

  Analyzing Segment 3...
    ✓ Found 588 rules for Segment 3

  Analyzing Segment 4...
    ✓ Found 716 rules for Segment 4


In [37]:
# ============================================================================
# PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT
# ============================================================================

print("\n" + "=" * 80)
print("PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT")
print("=" * 80)



PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT


In [38]:
# ============================================================================
# STEP 8.1: Top Products by Segment
# ============================================================================

print("\n[STEP 8.1] Identifying top products by segment...")

# Get top product families by segment
top_products_by_segment = df_trans_segments.groupby(['Cluster', 'LB_FAMILLE']).agg({
    'QT_UVC': 'sum',
    'MT_TTC_NET': 'sum',
    'CD_TICKET_UNIQUE': 'nunique'
}).reset_index()

top_products_by_segment.columns = ['Cluster', 'Product_Family', 'Total_Quantity', 'Total_Revenue', 'Unique_Transactions']
top_products_by_segment = top_products_by_segment.sort_values(['Cluster', 'Total_Revenue'], ascending=[True, False])

print("\nTop 5 Product Families by Segment:")
print("=" * 100)

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    segment_products = top_products_by_segment[top_products_by_segment['Cluster'] == cluster_id].head(5)
    print(f"\n📊 Segment {cluster_id}:")
    for idx, row in segment_products.iterrows():
        print(f"  {row['Product_Family']}: €{row['Total_Revenue']:,.0f} revenue | {row['Unique_Transactions']:,} transactions")

# Save
top_products_by_segment.to_csv('top_products_by_segment.csv', index=False)
print("\n✓ Top products saved to 'top_products_by_segment.csv'")



[STEP 8.1] Identifying top products by segment...

Top 5 Product Families by Segment:

📊 Segment 0:
  Poissons: €43,091 revenue | 3,718 transactions
  Tomate: €20,549 revenue | 4,269 transactions
  Oeufs: €19,060 revenue | 3,912 transactions
  Fromage méditerranéen: €17,482 revenue | 3,366 transactions
  Crustacés: €17,264 revenue | 2,263 transactions

📊 Segment 1:
  Poissons: €58,991 revenue | 2,785 transactions
  Tomate: €28,800 revenue | 3,163 transactions
  Raisin: €18,472 revenue | 2,828 transactions
  PPC: €17,561 revenue | 1,300 transactions
  Pâte fraîche: €17,293 revenue | 1,979 transactions

📊 Segment 2:
  Poissons: €15,024 revenue | 1,123 transactions
  Tomate: €7,918 revenue | 1,451 transactions
  Raisin: €7,138 revenue | 1,465 transactions
  Oeufs: €5,259 revenue | 1,002 transactions
  PPC: €4,548 revenue | 487 transactions

📊 Segment 3:
  Poissons: €23,772 revenue | 2,026 transactions
  Tomate: €17,480 revenue | 3,468 transactions
  Raisin: €15,907 revenue | 3,416 transa

In [10]:
# ============================================================================
# STEP 8.2: Cross-Selling Opportunities
# ============================================================================

print("\n[STEP 8.2] Identifying cross-selling opportunities...")

# For each segment, find products frequently bought together
cross_sell_opportunities = []

for cluster_id, rules_df in segment_rules_dict.items():
    if len(rules_df) > 0:
        # Get top cross-sell rules
        top_cross_sell = rules_df.nlargest(5, 'lift')
        
        for idx, rule in top_cross_sell.iterrows():
            cross_sell_opportunities.append({
                'Segment': cluster_id,
                'If_Customer_Buys': list(rule['antecedents']),
                'Recommend': list(rule['consequents']),
                'Confidence': rule['confidence'],
                'Lift': rule['lift']
            })

cross_sell_df = pd.DataFrame(cross_sell_opportunities)

if len(cross_sell_df) > 0:
    print("\nTop Cross-Selling Opportunities by Segment:")
    print("=" * 100)
    for cluster_id in cross_sell_df['Segment'].unique():
        segment_opps = cross_sell_df[cross_sell_df['Segment'] == cluster_id]
        print(f"\n🎯 Segment {cluster_id}:")
        for idx, row in segment_opps.iterrows():
            print(f"  If buys: {row['If_Customer_Buys']}")
            print(f"    → Recommend: {row['Recommend']} (Lift: {row['Lift']:.2f}, Confidence: {row['Confidence']:.1%})")
    
    cross_sell_df.to_csv('cross_sell_opportunities.csv', index=False)
    print("\n✓ Cross-sell opportunities saved to 'cross_sell_opportunities.csv'")
else:
    print("⚠ No cross-sell opportunities identified")



[STEP 8.2] Identifying cross-selling opportunities...

Top Cross-Selling Opportunities by Segment:

🎯 Segment 0:
  If buys: ['Poivron']
    → Recommend: ['Courgette'] (Lift: 3.85, Confidence: 34.5%)
  If buys: ['Courgette']
    → Recommend: ['Poivron'] (Lift: 3.85, Confidence: 28.1%)
  If buys: ['Carotte']
    → Recommend: ['Courgette'] (Lift: 3.58, Confidence: 32.1%)
  If buys: ['Courgette']
    → Recommend: ['Carotte'] (Lift: 3.58, Confidence: 36.3%)
  If buys: ['Poire']
    → Recommend: ['Fruits à noyau'] (Lift: 2.97, Confidence: 33.9%)

🎯 Segment 1:
  If buys: ['Aubergine', 'Tomate']
    → Recommend: ['Courgette', 'Poivron'] (Lift: 3.97, Confidence: 35.9%)
  If buys: ['Courgette', 'Poivron']
    → Recommend: ['Aubergine', 'Tomate'] (Lift: 3.97, Confidence: 29.7%)
  If buys: ['Poivron', 'Carotte']
    → Recommend: ['Condiment', 'Courgette', 'Tomate'] (Lift: 3.80, Confidence: 30.0%)
  If buys: ['Condiment', 'Courgette', 'Tomate']
    → Recommend: ['Poivron', 'Carotte'] (Lift: 3.80, 

In [42]:

# ============================================================================
# PART 10: BUSINESS RECOMMENDATIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 10: BUSINESS RECOMMENDATIONS & SUMMARY")
print("=" * 80)


PART 10: BUSINESS RECOMMENDATIONS & SUMMARY


In [43]:
# ============================================================================
# STEP 10.1: Segment Strategy Recommendations
# ============================================================================

print("\n[STEP 10.1] Generating segment-specific strategies...")

# Load segment profiles for recommendations
segment_profiles = pd.read_csv('segment_profiles.csv')
segment_business = pd.read_csv('segment_business_summary.csv')

recommendations = []

for cluster_id in segment_profiles['Cluster'].unique():
    profile = segment_profiles[segment_profiles['Cluster'] == cluster_id].iloc[0]
    business = segment_business[segment_business.index == cluster_id].iloc[0]
    
    # Build recommendation based on characteristics
    rec = {
        'Segment': cluster_id,
        'Size': f"{business['Customer_Count']:,.0f} customers ({business['Revenue_%']:.1f}% of revenue)",
        'Key_Characteristics': [],
        'Strategies': [],
        'Tactical_Actions': []
    }
    
    # Analyze characteristics
    if profile['recency_days'] < segment_profiles['recency_days'].median():
        rec['Key_Characteristics'].append("Recently active")
        rec['Strategies'].append("Maintain engagement")
    else:
        rec['Key_Characteristics'].append("At risk of churn")
        rec['Strategies'].append("Reactivation campaign")
        rec['Tactical_Actions'].append("Send personalized offers within 7 days")
    
    if profile['MT_TTC_NET_total_spend'] > segment_profiles['MT_TTC_NET_total_spend'].median():
        rec['Key_Characteristics'].append("High value")
        rec['Strategies'].append("VIP treatment & loyalty program")
        rec['Tactical_Actions'].append("Offer exclusive early access to promotions")
    
    if profile['promo_percentage'] > 50:
        rec['Key_Characteristics'].append("Price sensitive")
        rec['Strategies'].append("Targeted promotional campaigns")
        rec['Tactical_Actions'].append("Optimize promotional mix")
    
    if profile['LB_METIER_dept_diversity'] > segment_profiles['LB_METIER_dept_diversity'].median():
        rec['Key_Characteristics'].append("Diverse shopper")
        rec['Strategies'].append("Cross-category promotions")
    else:
        rec['Key_Characteristics'].append("Category specialist")
        rec['Strategies'].append("Deep assortment in preferred categories")
    
    if profile['store_loyalty_score'] > segment_profiles['store_loyalty_score'].median():
        rec['Key_Characteristics'].append("Store loyal")
        rec['Strategies'].append("Store-specific engagement")
    else:
        rec['Key_Characteristics'].append("Multi-store shopper")
        rec['Strategies'].append("Consistent experience across locations")
    
    recommendations.append(rec)

# Display recommendations
print("\n" + "=" * 100)
print("SEGMENT-SPECIFIC BUSINESS STRATEGIES")
print("=" * 100)

for rec in recommendations:
    print(f"\n{'='*100}")
    print(f"🎯 SEGMENT {rec['Segment']}")
    print(f"{'='*100}")
    print(f"Size: {rec['Size']}")
    print(f"\nKey Characteristics:")
    for char in rec['Key_Characteristics']:
        print(f"  • {char}")
    print(f"\nRecommended Strategies:")
    for strat in rec['Strategies']:
        print(f"  ► {strat}")
    if rec['Tactical_Actions']:
        print(f"\nTactical Actions:")
        for action in rec['Tactical_Actions']:
            print(f"  ✓ {action}")



[STEP 10.1] Generating segment-specific strategies...

SEGMENT-SPECIFIC BUSINESS STRATEGIES

🎯 SEGMENT 0
Size: 19,561 customers (28.8% of revenue)

Key Characteristics:
  • At risk of churn
  • Price sensitive
  • Category specialist
  • Multi-store shopper

Recommended Strategies:
  ► Reactivation campaign
  ► Targeted promotional campaigns
  ► Deep assortment in preferred categories
  ► Consistent experience across locations

Tactical Actions:
  ✓ Send personalized offers within 7 days
  ✓ Optimize promotional mix

🎯 SEGMENT 1
Size: 5,387 customers (30.9% of revenue)

Key Characteristics:
  • At risk of churn
  • High value
  • Price sensitive
  • Diverse shopper
  • Multi-store shopper

Recommended Strategies:
  ► Reactivation campaign
  ► VIP treatment & loyalty program
  ► Targeted promotional campaigns
  ► Cross-category promotions
  ► Consistent experience across locations

Tactical Actions:
  ✓ Send personalized offers within 7 days
  ✓ Offer exclusive early access to promotio

In [44]:
# ============================================================================
# STEP 10.2: Overall Business Recommendations
# ============================================================================

print("\n\n" + "=" * 100)
print("OVERALL BUSINESS RECOMMENDATIONS")
print("=" * 100)

overall_recommendations = """
1. CUSTOMER RETENTION & LOYALTY
   • Implement tiered loyalty program based on segment characteristics
   • Focus retention efforts on high-value segments showing increased recency
   • Create segment-specific communication cadences

2. PROMOTIONAL OPTIMIZATION
   • Reduce broad promotions; increase targeted offers by segment
   • Test national vs. store-level promotions by segment responsiveness
   • Optimize promotional calendar based on segment shopping patterns

3. CROSS-SELLING & UPSELLING
   • Implement recommendation engine using discovered association rules
   • Train staff on segment-specific cross-sell opportunities
   • Create bundled offers based on frequently co-purchased items

4. STORE-LEVEL STRATEGIES
   • Tailor product mix to dominant segments in each store
   • Adjust staffing and hours based on segment shopping times
   • Create store-specific marketing aligned with segment profiles

5. PRODUCT ASSORTMENT
   • Expand categories popular with high-value segments
   • Optimize inventory based on segment preferences by store
   • Test new products with most experimental segments first

6. CUSTOMER ACQUISITION
   • Profile prospects similar to best-performing segments
   • Tailor acquisition messaging to resonate with target segments
   • Focus marketing spend on attracting high-lifetime-value profiles

7. MEASUREMENT & OPTIMIZATION
   • Track segment migration over time (upgrades/downgrades)
   • Monitor segment-specific KPIs: retention rate, LTV, basket size
   • Refresh segmentation quarterly to capture behavioral changes
"""

print(overall_recommendations)




OVERALL BUSINESS RECOMMENDATIONS

1. CUSTOMER RETENTION & LOYALTY
   • Implement tiered loyalty program based on segment characteristics
   • Focus retention efforts on high-value segments showing increased recency
   • Create segment-specific communication cadences

2. PROMOTIONAL OPTIMIZATION
   • Reduce broad promotions; increase targeted offers by segment
   • Test national vs. store-level promotions by segment responsiveness
   • Optimize promotional calendar based on segment shopping patterns

3. CROSS-SELLING & UPSELLING
   • Implement recommendation engine using discovered association rules
   • Train staff on segment-specific cross-sell opportunities
   • Create bundled offers based on frequently co-purchased items

4. STORE-LEVEL STRATEGIES
   • Tailor product mix to dominant segments in each store
   • Adjust staffing and hours based on segment shopping times
   • Create store-specific marketing aligned with segment profiles

5. PRODUCT ASSORTMENT
   • Expand categories po

In [45]:
# ============================================================================
# STEP 10.3: Create Executive Summary
# ============================================================================

print("\n" + "=" * 100)
print("EXECUTIVE SUMMARY")
print("=" * 100)

# Calculate key metrics
total_customers = len(customer_segments)
total_revenue = customer_segments['MT_TTC_NET_total_spend'].sum()
num_segments = customer_segments['Cluster'].nunique()

# Identify best segment
best_segment = segment_business.loc[segment_business['Total_Revenue'].idxmax()]
best_segment_id = best_segment.name

summary = f"""
CUSTOMER SEGMENTATION ANALYSIS - KEY FINDINGS

Dataset Overview:
• Total Customers Analyzed: {total_customers:,}
• Total Revenue: €{total_revenue:,.0f}
• Number of Segments: {num_segments}
• Analysis Period: {df_transactions['DT_VENTE'].min()} to {df_transactions['DT_VENTE'].max()}

Segment Distribution:
"""

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    seg_data = segment_business.loc[cluster_id]
    summary += f"\n  Segment {cluster_id}: {seg_data['Customer_Count']:,} customers ({seg_data['Revenue_%']:.1f}% revenue)"

summary += f"""

Highest Value Segment:
• Segment {best_segment_id}
• {best_segment['Customer_Count']:,.0f} customers generating €{best_segment['Total_Revenue']:,.0f}
• Average revenue per customer: €{best_segment['Avg_Revenue_Per_Customer']:,.0f}

Key Opportunities:
• {len(cross_sell_opportunities)} cross-selling opportunities identified
• Segment-specific product recommendations developed
• Store-level optimization strategies defined

Next Steps:
1. Validate segment strategies with business stakeholders
2. Implement segment-based marketing campaigns
3. Deploy recommendation engine for cross-selling
4. Establish segment tracking and monitoring dashboard
5. Schedule quarterly segmentation refresh
"""

print(summary)

# Save summary
with open('executive_summary.txt', 'w') as f:
    f.write(summary)

print("\n✓ Executive summary saved to 'executive_summary.txt'")


EXECUTIVE SUMMARY

CUSTOMER SEGMENTATION ANALYSIS - KEY FINDINGS

Dataset Overview:
• Total Customers Analyzed: 48,309
• Total Revenue: €2,307,568
• Number of Segments: 5
• Analysis Period: 2023-09-11 to 2023-09-24

Segment Distribution:

  Segment 0: 19,561.0 customers (28.8% revenue)
  Segment 1: 5,387.0 customers (30.9% revenue)
  Segment 2: 1,810.0 customers (9.3% revenue)
  Segment 3: 7,909.0 customers (20.0% revenue)
  Segment 4: 13,642.0 customers (10.9% revenue)

Highest Value Segment:
• Segment 1
• 5,387 customers generating €712,905
• Average revenue per customer: €132

Key Opportunities:
• 25 cross-selling opportunities identified
• Segment-specific product recommendations developed
• Store-level optimization strategies defined

Next Steps:
1. Validate segment strategies with business stakeholders
2. Implement segment-based marketing campaigns
3. Deploy recommendation engine for cross-selling
4. Establish segment tracking and monitoring dashboard
5. Schedule quarterly segme

In [18]:
# ============================================================================
# FINAL OUTPUT SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE!")
print("=" * 100)

print("\n📊 Generated Outputs:")
outputs = [
    "1. customer_features_prepared.csv - Customer features for clustering",
    "2. customers_with_segments.csv - All customers with assigned segments",
    "3. segment_profiles.csv - Statistical profiles of each segment",
    "4. segment_business_summary.csv - Business metrics by segment",
    "5. association_rules_overall.csv - Overall association rules",
    "6. association_rules_segment_X.csv - Segment-specific rules",
    "7. top_products_by_segment.csv - Best-selling products per segment",
    "8. cross_sell_opportunities.csv - Cross-selling recommendations",
    "9. store_segment_distribution.csv - Segment mix by store",
    "10. store_performance_by_segment.csv - Store-segment performance",
    "11. executive_summary.txt - Executive summary report",
    "12. Various visualization PNG files"
]

for output in outputs:
    print(f"  {output}")

print("\n🎯 Ready for Business Implementation!")
print("=" * 100)
























ANALYSIS COMPLETE!

📊 Generated Outputs:
  1. customer_features_prepared.csv - Customer features for clustering
  2. customers_with_segments.csv - All customers with assigned segments
  3. segment_profiles.csv - Statistical profiles of each segment
  4. segment_business_summary.csv - Business metrics by segment
  5. association_rules_overall.csv - Overall association rules
  6. association_rules_segment_X.csv - Segment-specific rules
  7. top_products_by_segment.csv - Best-selling products per segment
  8. cross_sell_opportunities.csv - Cross-selling recommendations
  9. store_segment_distribution.csv - Segment mix by store
  10. store_performance_by_segment.csv - Store-segment performance
  11. executive_summary.txt - Executive summary report
  12. Various visualization PNG files

🎯 Ready for Business Implementation!
